Steps to preprocess VIIRS data for phenology tracking.

Author: Tayler LaSharr

Date: September 2024

In [123]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
import pandas as pd
import geemap
import os
import pprint

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

In [124]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '12-01'
end_month_day = '02-28'

## threshold for snow in winter to get the 'baseline' value in winter
snow_threshold = 50

## get the time period we care about (2023 data is still looking pretty weird for VIIRS)
start_date_data = '2018-01-01'
end_date_data = '2022-12-31'


Read in the different data pieces that we care about (VIIRS NDVI, MODIS snow, and mule deer polygons)

In [125]:
## load in a shapefile or area of interest
## this is an assest that is already loaded into GEE

# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## dry af
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()
small_square_geom = bounding_box  # Use bounding box directly as the clipping region

## load in the VIIRS data and the MODIS snow data
viirs_ndvi = ee.ImageCollection('NASA/VIIRS/002/VNP13A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDVI', 'VI_Quality') \
                .map(lambda image: image.clip(small_square_geom))

## right now i'm using the modis snow layer for processing snow, but VIIRS does have one, it's just not yet
## available for GEE

modis_snow = ee.ImageCollection('MODIS/006/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.clip(small_square_geom))

## first see the projection of the shapefile
coordidor_projection = md_cooridor.first().geometry().projection()
print('Projection of the cooridor shapefile:', coordidor_projection.getInfo())

## now we can reproject the shapefile
viirs_projection = viirs_ndvi.first().projection() ## this is the projection of the first image of the modis ndvi data
target_crs = viirs_projection.crs() ## this is the crs from the modis.projection object
target_crs_scale = viirs_projection.nominalScale() ## this is the scale (confused on what this is need to figure this out)

## this is a function to reproject it
def reproject_feature(feature):
    return feature.setGeometry(feature.geometry().transform(target_crs, target_crs_scale))

## this is running the function, so now "reprojected_roi" is the new object we care about
reprojected_roi = md_cooridor.map(reproject_feature)


Projection of the cooridor shapefile: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


Step 1. Remove low-quality observations with the QA band; this still might need some work. 

In [126]:
## define a function to only return high quality pixels

def quality_mask(image, snow_filter=True):
    ## select the VI_Quality band (not VI_Quality Bitmask if it's not present)
    quality = image.select('VI_Quality')

    ## MODLAND_QA (Bits 0-1) - keep only the pixels where value is 0 or 1 (good quality while also checking other QA things)
    modland_qa = quality.bitwiseAnd(3).lte(1)

    ## VI Usefulness (Bits 2-5) - keep pixels with values <= 2 (higher quality, highest values are worst and it goes to 15, I'm playing with this thershold)
    vi_usefulness = quality.rightShift(2).bitwiseAnd(15).lte(2)

    ## possible Snow/Ice (Bit 14) - mask pixels with snow or ice (might need to get rid of this to deal with winter later??)
    snow_ice = quality.rightShift(14).bitwiseAnd(1).eq(0)

    ## combine all the masks (can turn the snow mask on/off)
    if snow_filter:
        mask = modland_qa.And(vi_usefulness).And(snow_ice)
    else:
        mask = modland_qa.And(vi_usefulness)

    ## apply the mask to the image
    return image.updateMask(mask).select('NDVI')  ## select only NDVI after masking

## apply the mask to the VIIRS NDVI collection
viirs_ndvi_filtered = viirs_ndvi.map(lambda img: quality_mask(img, snow_filter=False))

Step 2. Set winter baselines dynamically since winter conditions are variable year to year. This differs from the Bischof methods because winter is not consistent and some winters have snow while others do not. For this step we basically go through 3 different substeps. First, we define the winter period, then we calculate the 'baseline' winter values during that period for all cells that have less than a set amount of snow cover. Finally, if there is snow covering the cells in winter, we will replace them with the 'winter baseline'. We won't set the winter baseline to the baseline for the entire winter because in desert systems there might be green that is more variable.

In [127]:
## function to add a property to the image for if it's winter

def filter_to_winter_period(image_collection, start_month_day, end_month_day):
    """Filters the image collection to the given winter time window."""
    
    ## function to check if the image falls within the specified winter period
    def filter_winter(image):
        image_date = ee.Date(image.get('system:time_start'))
        year = image_date.get('year')

        ## create date ranges for the winter period
        start_date = ee.Date.fromYMD(year, ee.Number.parse(start_month_day.split('-')[0]), ee.Number.parse(start_month_day.split('-')[1]))
        end_date = ee.Date.fromYMD(year.add(1), ee.Number.parse(end_month_day.split('-')[0]), ee.Number.parse(end_month_day.split('-')[1]))

        ## apply the date filter to determine if the image falls within the winter period
        is_winter = ee.Algorithms.If(
            image_date.millis().gte(start_date.millis()).And(image_date.millis().lte(end_date.millis())), 
            True, False
        )

        return image.set('is_winter', is_winter)

    ## apply the filter to the image collection and return only winter images
    winter_filtered = image_collection.map(filter_winter).filter(ee.Filter.eq('is_winter', True))

    return winter_filtered


## run the function
viirs_ndvi_winter = filter_to_winter_period(viirs_ndvi, start_month_day, end_month_day)


In [128]:
## now a function to mask the pixels that have snow above the threshold we care about so we can calculate the 'winter baseline'. this is using
## the entire period, but because VIIRS does not have 9 year data is going to be a shorter time period than the 9 year window that Bischof uses

def mask_snow_using_modis(image, snow_threshold=50):
    
    ## get the VIIRS image date
    viirs_date = ee.Date(image.get('system:time_start'))

    ## get the closest MODIS snow image before and after the VIIRS image date
    modis_snow_image_before = modis_snow.filterDate(start_date_data, viirs_date).sort('system:time_start', False).first()
    modis_snow_image_after = modis_snow.filterDate(viirs_date, end_date_data).sort('system:time_start').first()

    modis_snow_image = ee.Algorithms.If(
        viirs_date.difference(ee.Date(modis_snow_image_before.get('system:time_start')), 'day').abs().lte(
        viirs_date.difference(ee.Date(modis_snow_image_after.get('system:time_start')), 'day').abs()),
        modis_snow_image_before, modis_snow_image_after
    )

    ## mask pixels where snow cover is greater than the threshold
    snow_mask = ee.Image(modis_snow_image).select('NDSI_Snow_Cover').gt(snow_threshold)
    
    ## apply the snow mask to the NDVI image (keep pixels with snow cover below the threshold)
    return image.updateMask(snow_mask.Not())


## apply the snow mask to the winter period NDVI collection
viirs_ndvi_winter_no_snow = viirs_ndvi_winter.map(lambda img: mask_snow_using_modis(img, snow_threshold))


## then calculate the 0.025 percentile of NDVI values in the snow-masked winter NDVI collection
def calculate_baseline(image_collection, quantile=0.025):
    
    ## reduce the image collection to get the 0.025 percentile
    baseline_ndvi = image_collection.select('NDVI').reduce(ee.Reducer.percentile([quantile * 100])).rename('NDVI_baseline')
    return baseline_ndvi

## calculate the baseline for the winter period
winter_baseline_ndvi = calculate_baseline(viirs_ndvi_winter_no_snow, quantile=0.025)



In [129]:
def apply_baseline_correction(image_collection, baseline_image, snow_threshold=50, window_size=2, modis_snow=None):
    """
    Adjust NDVI values based on baseline for snow-covered images and apply a moving window.
    
    Args:
    - image_collection: The collection of NDVI images.
    - baseline_image: The baseline NDVI image calculated for winter.
    - snow_threshold: The snow cover threshold for masking (default is 50%).
    - window_size: The number of surrounding images to check for snow cover (default is 2 images).
    - modis_snow: MODIS snow collection to use for snow cover masking.
    
    Returns:
    - corrected_collection: The image collection with NDVI values adjusted based on the baseline and snow cover.
    """
    
    ## function to get the closest MODIS snow image to the VIIRS image
    def get_closest_modis_snow(image_date):
        modis_snow_before = modis_snow.filterDate('2018-01-01', image_date).sort('system:time_start', False).first()
        modis_snow_after = modis_snow.filterDate(image_date, '2022-12-31').sort('system:time_start').first()

        ## choose the closest MODIS image (either before or after)
        modis_snow_image = ee.Algorithms.If(
            image_date.difference(ee.Date(modis_snow_before.get('system:time_start')), 'day').abs().lte(
            image_date.difference(ee.Date(modis_snow_after.get('system:time_start')), 'day').abs()),
            modis_snow_before, modis_snow_after
        )

        return ee.Image(modis_snow_image)
    
    # function to adjust NDVI for snow-covered pixels
    def adjust_ndvi(image):
        image_date = ee.Date(image.get('system:time_start'))

        ## get the closest MODIS snow image
        modis_snow_image = get_closest_modis_snow(image_date)
        
        ## create a snow mask where snow cover exceeds the threshold
        snow_mask = modis_snow_image.select('NDSI_Snow_Cover').gt(snow_threshold)
        
        ## apply the baseline NDVI where snow mask is true, keep original NDVI otherwise
        adjusted_ndvi = image.select('NDVI').where(snow_mask, baseline_image.select('NDVI_baseline'))
        
        return image.addBands(adjusted_ndvi.rename('NDVI'), overwrite=True)
    
    ## apply the initial snow correction to the entire image collection
    corrected_collection = image_collection.map(adjust_ndvi)

    
    ## function to apply the moving window logic for snow-covered pixels
    def apply_moving_window(image):
        image_date = ee.Date(image.get('system:time_start'))
        
        ## get surrounding images within the moving window
        surrounding_images = image_collection.filter(ee.Filter.date(
            image_date.advance(-window_size, 'month'), image_date.advance(window_size, 'month')))
        
        ## for each surrounding image, check snow cover and return the snow mask
        surrounding_snow_masks = surrounding_images.map(lambda img: 
            get_closest_modis_snow(ee.Date(img.get('system:time_start')))
            .select('NDSI_Snow_Cover').gt(snow_threshold)
        )
        
        ## combine the surrounding snow masks using a logical "or" operation
        moving_window_mask = surrounding_snow_masks.reduce(ee.Reducer.anyNonZero())
        
        ## apply the baseline NDVI where the moving window mask is true
        adjusted_ndvi_window = image.select('NDVI').where(moving_window_mask, baseline_image.select('NDVI_baseline'))
        
        return image.addBands(adjusted_ndvi_window.rename('NDVI'), overwrite=True)
    
    ## apply the moving window correction to the collection
    corrected_collection_with_window = corrected_collection.map(apply_moving_window)
    
    return corrected_collection_with_window



## apply the baseline correction and moving window (to set the winter min)
viirs_ndvi_corrected = apply_baseline_correction(viirs_ndvi_filtered,
                                                 winter_baseline_ndvi,
                                                 snow_threshold=50,
                                                 window_size=2,
                                                 modis_snow=modis_snow)



Step 3. Apply a smoothing function (moving median window with a search window of 3)

In [130]:
def apply_image_median(image_collection, window_size=3):
    
    ## define a function to apply median filtering using a fixed number of images around each image
    def smooth_image(current_image):
        
        ## get the list of all images in the collection, sorted by date
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())  # Convert to a list

        ## get the index of the current image
        image_index = all_images_list.indexOf(current_image)
        
        ## calculate half of the window size
        half_window = ee.Number(window_size // 2)

        ## define the start and end indices for the window (centered on the current image)
        ## ensure start_index is not less than 0
        start_index = ee.Number(image_index).subtract(half_window).max(0)

        ## ensure end_index is not greater than the total number of images
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## create a list of images within the window
        window_images = all_images_list.slice(start_index, end_index.add(1))  # Include the end image
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median of the selected images
        median_image = window_collection.median()

        ## return the median-smoothed image, keeping the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    ## apply the smoothing function over the entire image collection
    smoothed_collection = image_collection.map(smooth_image)

    return smoothed_collection

## apply the smoothing function to the processed_ndvi collection
smoothed_ndvi = apply_image_median(viirs_ndvi_corrected, window_size=3)


In [131]:
def export_collection_as_stack(image_collection, file_name, scale=500):
    try:

        ## cast to float32
        image_collection = image_collection.map(lambda img: img.toFloat())

        ## stack all images as bands in a single multi-band image
        stacked_image = image_collection.toBands()

        ## export the stacked image to Google Drive
        task = ee.batch.Export.image.toDrive(
            image=stacked_image,
            description=f'{file_name}_stack',
            folder='DroughtSurfing',  # specify the folder in your Google Drive 
            fileNamePrefix=file_name,  # name for the exported file
            scale=scale,  # resolution in meters
            region=small_square_geom,  # region to export
            maxPixels=1e13,  # set to a high value to avoid pixel limit issues
            fileFormat='GeoTIFF'  # export as GeoTIFF
        )
        task.start()
        print(f'Exporting {file_name}_stack to Google Drive...')
    except Exception as e:
        print(f'Error exporting {file_name}: {e}')

export_collection_as_stack(viirs_ndvi.select('NDVI'), 'viirs_ndvi') ## because this still has the quality band, just export the NDVI band to the rasterstack
export_collection_as_stack(viirs_ndvi_corrected, 'viirs_ndvi_corrected')
export_collection_as_stack(viirs_ndvi_filtered, 'viirs_ndvi_filtered')
export_collection_as_stack(smoothed_ndvi, 'smoothed_ndvi')

## takes about 20 minutes to export the images. Open up GEE (https://code.earthengine.google.com/) to see if it's working and it will
## give you errors if it doesn't work.



Exporting viirs_ndvi_stack to Google Drive...
Exporting viirs_ndvi_corrected_stack to Google Drive...
Exporting viirs_ndvi_filtered_stack to Google Drive...
Exporting smoothed_ndvi_stack to Google Drive...
